In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
column_names = [
    "duration","protocol_type","service","flag","src_bytes","dst_bytes",
    "land","wrong_fragment","urgent","hot","num_failed_logins","logged_in",
    "num_compromised","root_shell","su_attempted","num_root",
    "num_file_creations","num_shells","num_access_files","num_outbound_cmds",
    "is_host_login","is_guest_login","count","srv_count","serror_rate",
    "srv_serror_rate","rerror_rate","srv_rerror_rate","same_srv_rate",
    "diff_srv_rate","srv_diff_host_rate","dst_host_count",
    "dst_host_srv_count","dst_host_same_srv_rate","dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate","dst_host_srv_diff_host_rate",
    "dst_host_serror_rate","dst_host_srv_serror_rate","dst_host_rerror_rate",
    "dst_host_srv_rerror_rate","label","difficulty"
]

train_path = "data/raw/KDDTrain+.txt"
test_path = "data/raw/KDDTest+.txt"

df_train = pd.read_csv(train_path, header=None, names=column_names)
df_test = pd.read_csv(test_path, header=None, names=column_names)

df_train.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


In [2]:
df_train.shape
df_train.columns

Index(['duration', 'protocol_type', 'service', 'flag', 'src_bytes',
       'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot',
       'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell',
       'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
       'num_access_files', 'num_outbound_cmds', 'is_host_login',
       'is_guest_login', 'count', 'srv_count', 'serror_rate',
       'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
       'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count',
       'dst_host_srv_count', 'dst_host_same_srv_rate',
       'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
       'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
       'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
       'dst_host_srv_rerror_rate', 'label', 'difficulty'],
      dtype='object')

In [3]:
#Define attack lists and mapping function
dos_attacks = [
    'back', 'land', 'neptune', 'pod', 'smurf', 'teardrop',
    'apache2', 'mailbomb', 'processtable', 'udpstorm', 'worm',
    'buffer_overflow'
]

probe_attacks = [
    'satan', 'ipsweep', 'nmap', 'portsweep', 'mscan', 'saint'
]

r2l_attacks = [
    'guess_passwd', 'ftp_write', 'imap', 'phf', 'multihop',
    'warezmaster', 'warezclient', 'spy', 'xlock', 'xsnoop',
    'snmpgetattack', 'snmpguess', 'httptunnel', 'sendmail', 'named'
]

u2r_attacks = [
    'rootkit', 'perl', 'loadmodule', 'ps', 'sqlattack'
]

def map_attack(label: str) -> str:
    """
    Map a raw NSL-KDD attack label to one of:
    'DoS', 'Probe', 'R2L', 'U2R', 'normal'.
    """
    if label == 'normal':
        return 'normal'
    elif label in dos_attacks:
        return 'DoS'
    elif label in probe_attacks:
        return 'Probe'
    elif label in r2l_attacks:
        return 'R2L'
    elif label in u2r_attacks:
        return 'U2R'
    else:
        # For any unexpected label, you can either return 'normal'
        # or 'unknown'; here we mark it as 'unknown' so you can spot it.
        return 'unknown'



In [4]:
# Create attack_category column and check values
# 2) Apply the mapping to train and test data

df_train['attack_category'] = df_train['label'].apply(map_attack)
df_test['attack_category'] = df_test['label'].apply(map_attack)

# Look at first 10 rows: raw label vs new category
df_train[['label', 'attack_category']].head(10)


,label,attack_category
0,normal,normal
1,normal,normal
2,neptune,DoS
3,normal,normal
4,normal,normal
5,neptune,DoS
6,neptune,DoS
7,neptune,DoS
8,neptune,DoS
9,neptune,DoS


In [5]:
#Drop difficulty, remove duplicates, define column groups
# Drop 'difficulty' column for now
df_train = df_train.drop(columns=['difficulty'])
df_test = df_test.drop(columns=['difficulty'])

# Remove duplicate rows
print("Duplicates before:", df_train.duplicated().sum())
df_train = df_train.drop_duplicates()
print("Duplicates after :", df_train.duplicated().sum())

# Identify column types
categorical_cols = ['protocol_type', 'service', 'flag']
label_cols = ['label', 'attack_category']
feature_cols = [c for c in df_train.columns if c not in label_cols]
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

categorical_cols, numeric_cols


Duplicates before: 0
Duplicates after : 0


(['protocol_type', 'service', 'flag'],
 ['duration',
  'src_bytes',
  'dst_bytes',
  'land',
  'wrong_fragment',
  'urgent',
  'hot',
  'num_failed_logins',
  'logged_in',
  'num_compromised',
  'root_shell',
  'su_attempted',
  'num_root',
  'num_file_creations',
  'num_shells',
  'num_access_files',
  'num_outbound_cmds',
  'is_host_login',
  'is_guest_login',
  'count',
  'srv_count',
  'serror_rate',
  'srv_serror_rate',
  'rerror_rate',
  'srv_rerror_rate',
  'same_srv_rate',
  'diff_srv_rate',
  'srv_diff_host_rate',
  'dst_host_count',
  'dst_host_srv_count',
  'dst_host_same_srv_rate',
  'dst_host_diff_srv_rate',
  'dst_host_same_src_port_rate',
  'dst_host_srv_diff_host_rate',
  'dst_host_serror_rate',
  'dst_host_srv_serror_rate',
  'dst_host_rerror_rate',
  'dst_host_srv_rerror_rate'])

In [6]:
#Create numeric targets (binary and 5‑class)
# Encode labels and categorical features
import numpy as np
from sklearn.preprocessing import LabelEncoder

# Binary label (attack vs normal)
df_train['binary_label'] = np.where(df_train['attack_category'] == 'normal', 0, 1)
df_test['binary_label'] = np.where(df_test['attack_category'] == 'normal', 0, 1)

# 5-class label (including 'unknown' if present) – fit on both train and test
le_attack = LabelEncoder()
le_attack.fit(pd.concat([df_train['attack_category'], df_test['attack_category']], axis=0))

df_train['attack_cat_id'] = le_attack.transform(df_train['attack_category'])
df_test['attack_cat_id'] = le_attack.transform(df_test['attack_category'])

df_train[['attack_category', 'attack_cat_id', 'binary_label']].head()


,attack_category,attack_cat_id,binary_label
0,normal,4,0
1,normal,4,0
2,DoS,0,1
3,normal,4,0
4,normal,4,0


In [7]:
#One‑hot encode categorical features
df_train_enc = pd.get_dummies(df_train, columns=['protocol_type', 'service', 'flag'])
df_test_enc = pd.get_dummies(df_test, columns=['protocol_type', 'service', 'flag'])

# Align columns between train/test
df_train_enc, df_test_enc = df_train_enc.align(df_test_enc, join='left', axis=1, fill_value=0)

df_train_enc.shape, df_test_enc.shape


((125973, 126), (22544, 126))

In [8]:
#Scale numeric features and make X/y
from sklearn.preprocessing import StandardScaler

target_col = 'attack_cat_id'

X_train = df_train_enc.drop(columns=['label','attack_category','binary_label','attack_cat_id'])
y_train = df_train_enc[target_col]

X_test = df_test_enc.drop(columns=['label','attack_category','binary_label','attack_cat_id'])
y_test = df_test_enc[target_col]

numeric_in_X = [c for c in X_train.columns if c in numeric_cols]

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numeric_in_X] = scaler.fit_transform(X_train[numeric_in_X])
X_test_scaled[numeric_in_X] = scaler.transform(X_test[numeric_in_X])

X_train_scaled.shape, X_test_scaled.shape


((125973, 122), (22544, 122))

In [9]:
#Grid search for RandomForest
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 20, 40],
    'min_samples_split': [2, 5]
}

rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)

grid_rf = GridSearchCV(
    rf_base,
    param_grid,
    cv=3,
    scoring='f1_macro',
    n_jobs=-1
)

grid_rf.fit(X_train_scaled, y_train)

print("Best params:", grid_rf.best_params_)
print("Best CV macro F1:", grid_rf.best_score_)


Best params: {'max_depth': 40, 'min_samples_split': 2, 'n_estimators': 100}
Best CV macro F1: 0.8349837002363177
